# MRI_Detection - Step 1: Data overview
**Goal:** unpack the Kaggle *Labeled MRI Brain Tumor Dataset* (`archive.zip`), audit the YOLO labels and look at the data before touching any model.

Put `archive.zip` in `data/raw/` first.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.ROOT / "runs"
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = YOLO_DIR / "data.yaml"

## 1.1 Extract the archive

In [ ]:
root = data.extract_zip(md.ROOT / cfg["dataset"]["raw_zip"], md.ROOT / cfg["dataset"]["raw_dir"])
print("Dataset root:", root)

## 1.2 Scan every image and its label file
Each class folder holds `image.jpg` + `image.txt` (YOLO format: `class x_center y_center width height`, normalised).

In [ ]:
df = data.scan_dataset(root)
print("Images:", len(df))
df.groupby("folder").agg(images=("image", "size"), boxes=("n_boxes", "sum"))

## 1.3 Label-quality audit
Things the audit looks for: missing or empty label files, malformed rows, boxes outside [0,1], labels whose class disagrees with the image's folder, byte-identical duplicate images, and stray `.txt` files without an image.

In [ ]:
df, folder2id, names = data.clean_dataset(df)
print("Class id per folder:", folder2id)
print("Class names (ordered by id):", names)
print("\nStray label-like files with no image:", data.find_orphan_labels(root))
print("\nImages with a label problem:")
bad = df[df.issue.notna()][["image", "folder", "issue"]].copy()
bad["image"] = bad.image.map(lambda p: Path(p).name)
display(bad)
dups = df[df.duplicated("md5", keep=False)]
print(f"\nExact-duplicate images: {dups.md5.nunique()} groups covering {len(dups)} images")

**What the audit means.** The class ids come from the labels themselves (the dataset has no `data.yaml`), so they are inferred per folder: the most common id in a folder is that folder's class. Images with a label problem are *quarantined* (excluded from training) in step 2. Duplicate images are not removed but are kept in the same split so they cannot leak between train and validation.

## 1.4 Class balance and box sizes

In [ ]:
import numpy as np, matplotlib.pyplot as plt
ok = df[df.issue.isna()]
boxes = [(names[b[0]], b[3], b[4]) for bx in ok.boxes for b in bx]
import pandas as pd
bdf = pd.DataFrame(boxes, columns=["class", "w", "h"]); bdf["area"] = bdf.w * bdf.h
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
bdf["class"].value_counts().reindex(names).plot.bar(ax=ax[0], title="Boxes per class")
ok.n_boxes.value_counts().sort_index().plot.bar(ax=ax[1], title="Boxes per image")
ax[2].hist(bdf.area, bins=40); ax[2].set_title("Box area (fraction of image)")
plt.tight_layout(); plt.show()
print(bdf.groupby("class")[["w", "h", "area"]].mean().round(3))
from PIL import Image
sizes = pd.Series([Image.open(p).size for p in ok.image.sample(300, random_state=0)]).value_counts().head(5)
print("\nMost common image sizes (w, h), 300-image sample:\n", sizes)

Note: the `notumor` class is labelled with a large box covering most of the brain, so the detector learns "no tumour" as a fourth class rather than predicting nothing.

## 1.5 Look at the images with their boxes

In [ ]:
import random
random.seed(0)
picks = []
for n in names:
    picks += random.sample(list(ok[ok.folder == n].index), 3)
imgs = [viz.draw_boxes(ok.loc[i, "image"], ok.loc[i, "boxes"], names) for i in picks]
viz.show_grid(imgs, ncols=4, figsize=(14, 3.6), titles=[ok.loc[i, "folder"] for i in picks]); plt.show()

Next: `02_dataset_split.ipynb`.